# Classify using DTW on stretched spikes

In [ ]:
cut_params = {
    'signal_name': 'wideband',
    'event_name': 'spk',
    'cut_window': [-0.001, 0.002],
    'fs': 40000
}
proc_params = {
    'smooth_params': {
        'window_length': 9,
        'polyorder': 3
    },
    'dv_threshold': 0.1
}

In [ ]:
import os

INVITRO_DATA_DIR = f'{os.environ["RAAP_INVITRO_DATA"]}/juxta'
INVIVO_DATA_DIR = f'{os.environ["RAAP_INVIVO_DATA"]}'
INVIVO_FEATURE_DIR = f'{os.environ["RAAP_INVIVO_FEATURES"]}'

In [ ]:
training_cell_names = [
    '2406241PZ_01',
    '2406242PZ_03',
    '2406263PZ_01',
    '2406282PZ_01',
    '2407031PZ_01',
    '2407031SI_gr2',
    '2407041SI_gr1'
]

In [ ]:
from workflow import load_training_dataset
training_dataset = load_training_dataset(
    training_cell_names,
    INVITRO_DATA_DIR,
    cut_params=cut_params,
    proc_params=proc_params
)

In [ ]:
from transformers import (
    rising_phase,
    phase_curve,
    normalize_phase_curve_range,
    resample_norm_phase_curve
)
test_ais = training_dataset[0]['AISAP'][0]
test_ra = training_dataset[0]['RAAP'][0]
ais_phase = phase_curve(rising_phase(test_ais))
ra_phase = phase_curve(rising_phase(test_ra))
ais_phase_norm = normalize_phase_curve_range(ais_phase)
ra_phase_norm = normalize_phase_curve_range(ra_phase)
ais_phase_resampled = resample_norm_phase_curve(ais_phase_norm, 20)
ra_phase_resampled = resample_norm_phase_curve(ra_phase_norm, 20)

In [ ]:
from dtw import dtw
raw_alignment = dtw(
    ais_phase.as_array(),
    ra_phase.as_array(),
    dist_method='euclidean'
)
norm_alignment = dtw(
    ais_phase_norm.as_array(),
    ra_phase_norm.as_array(),
    dist_method='euclidean'
)
resampled_alignment = dtw(
    ais_phase_resampled.as_array(),
    ra_phase_resampled.as_array(),
    dist_method='euclidean'
)
unnorm_dist = getattr(raw_alignment, 'distance')
norm_dist = getattr(norm_alignment, 'normalizedDistance')
resampled_dist = getattr(resampled_alignment, 'distance')
print(f'Unnormalized DTW distance: {unnorm_dist}')
print(f'Normalized DTW distance: {norm_dist}')
print(f'Resampled DTW distance: {resampled_dist}')

In [ ]:
import matplotlib.pyplot as plt
fig, axes = plt.subplots(1, 2, figsize=(6, 6))
axes[0].plot(
    ais_phase.v,
    ais_phase.dvdt,
    color='C0'
)
axes[0].plot(
    ra_phase.v,
    ra_phase.dvdt,
    color='C1'
)
axes[1].plot(
    ais_phase_norm.v,
    ais_phase_norm.dvdt,
    color='C0'
)
axes[1].plot(
    ais_phase_resampled.v,
    ais_phase_resampled.dvdt,
    '--',
    color='black'
)
axes[1].plot(
    ra_phase_norm.v,
    ra_phase_norm.dvdt,
    color='C1'
)
axes[1].plot(
    ra_phase_resampled.v,
    ra_phase_resampled.dvdt,
    '--',
    color='black'
)
plt.show()

In [ ]:
import numpy as np
n_training_samples = len(training_dataset)
curves, y = [], []
for i, spike_type in enumerate(['AISAP', 'RAAP']):
    for cell_shapes in training_dataset:
        for spike in cell_shapes[spike_type]:
            phase = resample_norm_phase_curve(
                normalize_phase_curve_range(
                    phase_curve(rising_phase(spike))
                ), 20
            )
            curves.append(phase)
            y.append(i)
y = np.array(y)
n_spikes = len(curves)

In [ ]:
from tqdm import tqdm
D = np.zeros((n_spikes, n_spikes))
for i in tqdm(range(n_spikes), total=n_spikes):
    for j in range(i + 1, n_spikes):
        alignment = dtw(
            curves[i].as_array(),
            curves[j].as_array(),
            dist_method='euclidean'
        )
        D[i, j] = getattr(alignment, 'distance')
        D[j, i] = D[i, j]

In [ ]:
from dtw_classifier.dataclasses import PhaseCurve
group_mean_curves = []
for yy in [0, 1]:
    group_curves = [curve for curve, label in zip(curves, y) if label == yy]
    group_mean_curves.append(
        PhaseCurve(
            np.mean([curve.v for curve in group_curves], axis=0),
            np.mean([curve.dvdt for curve in group_curves], axis=0)
        )
    )

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(6, 6))
for curve, label in zip(curves, y):
    axes[label].plot(
        group_mean_curves[label].v,
        group_mean_curves[label].dvdt,
        color='C0' if label == 0 else 'C1',
        alpha=0.3
    )
plt.show()

In [ ]:
import pandas as pd
D_from_mean = np.zeros((n_spikes, 2))
for i, curve in tqdm(enumerate(curves), total=n_spikes):
    for yy in [0, 1]:
        alignment = dtw(
            curve.as_array(),
            group_mean_curves[yy].as_array(),
            dist_method='euclidean'
        )
        D_from_mean[i, yy] = getattr(alignment, 'distance')
print(D_from_mean.shape, y.shape)

mean_dist_df = pd.DataFrame(np.stack([D_from_mean[:, 0], D_from_mean[:, 1], y], axis=1), columns=['AISAP', 'RAAP', 'Label'])
mean_dist_df['id'] = mean_dist_df.index
mean_dist_df['pref'] = (
    mean_dist_df['AISAP'] - mean_dist_df['RAAP']
) / (
    mean_dist_df['AISAP'] + mean_dist_df['RAAP']
)

In [ ]:
import seaborn as sns
fig, ax = plt.subplots(figsize=(6, 6))
sns.scatterplot(
    data=mean_dist_df,
    x='id',
    y='pref',
    hue='Label',
    ax=ax
)
plt.show()

In [ ]:
mean_dist_df[
    (mean_dist_df['pref'] > 0) & (mean_dist_df['Label'] == 0)
]

In [ ]:
fig, axes = plt.subplots(4, 4, figsize=(12, 12))
i = 0
for _, row in mean_dist_df[
    (mean_dist_df['pref'] > 0) & (mean_dist_df['Label'] == 0)
].iterrows():
    curve = curves[int(row['id'])]
    axes[i // 4, i % 4].plot(
        curve.v,
        curve.dvdt,
        color='C0' if row['Label'] == 0 else 'C1'
    )
    i += 1
plt.show()

In [ ]:
fig, axes = plt.subplots(10, 10, figsize=(12, 12))
i = 0
for _, row in mean_dist_df[
    (mean_dist_df['pref'] < 0) & (mean_dist_df['Label'] == 1)
].iterrows():
    curve = curves[int(row['id'])]
    axes[i // 10, i % 10].plot(
        curve.v,
        curve.dvdt,
        color='C0' if row['Label'] == 0 else 'C1'
    )
    i += 1
plt.show()

In [ ]:
within_class_distances = []
between_class_distances = []
for yy in [0, 1]:
    flag = (y == yy)
    D_class = D_from_mean[flag, yy]
    within_class_distances.append((np.mean(D_class), np.std(D_class)))
    between_class_distances.append((np.mean(D_from_mean[flag, 1 - yy]), np.std(D_from_mean[flag, 1 - yy])))
print(f'Mean within-class distance for AISAP: {within_class_distances[0]}')
print(f'Mean within-class distance for RAAP: {within_class_distances[1]}')
print(f'Mean between-class distance for AISAP: {between_class_distances[0]}')
print(f'Mean between-class distance for RAAP: {between_class_distances[1]}')